# Entrenamiento LoRA de Stable Diffusion 1.5 para semillas de soja

Notebook corregido para Google Colab.

## Objetivo

1. Verificar GPU.
2. Montar Google Drive.
3. Instalar Diffusers sin reemplazar PyTorch CUDA.
4. Corregir la incompatibilidad de `torchao`.
5. Extraer y validar `soybean_lora_dataset.zip`.
6. Ejecutar un smoke test de 50 pasos.
7. Entrenar el LoRA completo.
8. Guardar el adaptador en Drive.
9. Comparar Stable Diffusion 1.5 base contra Stable Diffusion 1.5 + LoRA.

> El entrenamiento anterior no produjo pesos porque falló antes de comenzar por una versión incompatible de `torchao`. Este notebook detiene la ejecución inmediatamente cuando una etapa falla.


## 1. Comprobar GPU

In [1]:
import subprocess
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA compilada:", torch.version.cuda)
print("CUDA disponible:", torch.cuda.is_available())

subprocess.run(["nvidia-smi"], check=False)

assert torch.cuda.is_available(), (
    "No hay GPU CUDA disponible. En Colab selecciona "
    "Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU."
)

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB",
)


Python: 3.12.13
PyTorch: 2.11.0+cu128
CUDA compilada: 12.8
CUDA disponible: True
GPU: Tesla T4
VRAM: 14.56 GB


## 2. Montar Google Drive

In [2]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 3. Definir todas las rutas

In [3]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SeedCare-RAG-LoRA")
ZIP_PATH = DRIVE_ROOT / "datasets" / "soybean_lora_dataset.zip"

WORK_ROOT = Path("/content/seedcare")
DATASET_DIR = WORK_ROOT / "dataset"
DIFFUSERS_DIR = WORK_ROOT / "diffusers"
OUTPUTS_DIR = WORK_ROOT / "outputs"

SMOKE_OUTPUT_DIR = OUTPUTS_DIR / "smoke_test"
LORA_OUTPUT_DIR = OUTPUTS_DIR / "soybean_sd15_lora"
COMPARISON_DIR = WORK_ROOT / "comparisons"

DRIVE_MODEL_DIR = DRIVE_ROOT / "models" / "soybean_sd15_lora"
DRIVE_COMPARISON_DIR = DRIVE_ROOT / "outputs" / "comparisons"

TRAIN_SCRIPT = (
    DIFFUSERS_DIR
    / "examples"
    / "text_to_image"
    / "train_text_to_image_lora.py"
)

MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
WEIGHT_NAME = "pytorch_lora_weights.safetensors"

for directory in [
    WORK_ROOT,
    OUTPUTS_DIR,
    COMPARISON_DIR,
    DRIVE_MODEL_DIR,
    DRIVE_COMPARISON_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("ZIP:", ZIP_PATH)
print("Dataset temporal:", DATASET_DIR)
print("Salida smoke test:", SMOKE_OUTPUT_DIR)
print("Salida LoRA final:", LORA_OUTPUT_DIR)
print("Destino permanente:", DRIVE_MODEL_DIR)

assert ZIP_PATH.exists(), f"No se encontró el ZIP: {ZIP_PATH}"


ZIP: /content/drive/MyDrive/SeedCare-RAG-LoRA/datasets/soybean_lora_dataset.zip
Dataset temporal: /content/seedcare/dataset
Salida smoke test: /content/seedcare/outputs/smoke_test
Salida LoRA final: /content/seedcare/outputs/soybean_sd15_lora
Destino permanente: /content/drive/MyDrive/SeedCare-RAG-LoRA/models/soybean_sd15_lora


## 4. Instalar Diffusers y dependencias

Se desinstala `torchao` porque la versión preinstalada observada (`0.10.0`) era incompatible con PEFT y bloqueaba la creación del adaptador LoRA. `torchao` no es necesario para este entrenamiento.


In [4]:
import os
import shutil
import subprocess

# No reinstalar torch: Colab ya proporcionó una compilación con CUDA.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=False,
)

if DIFFUSERS_DIR.exists():
    shutil.rmtree(DIFFUSERS_DIR)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/huggingface/diffusers.git",
        str(DIFFUSERS_DIR),
    ],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", str(DIFFUSERS_DIR)],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(
            DIFFUSERS_DIR
            / "examples"
            / "text_to_image"
            / "requirements.txt"
        ),
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "bitsandbytes",
        "tensorboard",
    ],
    check=True,
)

assert TRAIN_SCRIPT.exists(), f"No existe el script: {TRAIN_SCRIPT}"
assert torch.cuda.is_available(), "CUDA dejó de estar disponible."

print("Instalación terminada.")


Instalación terminada.


## 5. Verificar versiones

In [5]:
import importlib.metadata as metadata

packages = [
    "torch",
    "torchvision",
    "diffusers",
    "transformers",
    "accelerate",
    "peft",
    "datasets",
    "bitsandbytes",
]

for package in packages:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: no instalado")

try:
    print("torchao:", metadata.version("torchao"))
except metadata.PackageNotFoundError:
    print("torchao: no instalado (correcto para este notebook)")

print("CUDA disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


torch: 2.11.0+cu128
torchvision: 0.26.0+cu128
diffusers: 0.40.0.dev0
transformers: 5.12.1
accelerate: 1.14.0
peft: 0.19.1
datasets: 4.0.0
bitsandbytes: 0.49.2
torchao: no instalado (correcto para este notebook)
CUDA disponible: True
GPU: Tesla T4


## 6. Configurar Accelerate

In [6]:
subprocess.run(["accelerate", "config", "default"], check=True)


CompletedProcess(args=['accelerate', 'config', 'default'], returncode=0)

## 7. Extraer el dataset limpiamente

La extracción normaliza las barras invertidas creadas por Windows y elimina previamente la copia temporal anterior. El ZIP almacenado en Drive no se modifica.


In [7]:
import zipfile

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

DATASET_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    for member in archive.infolist():
        normalized_name = member.filename.replace("\\", "/").lstrip("/")
        if not normalized_name:
            continue

        destination = (DATASET_DIR / normalized_name).resolve()
        dataset_root = DATASET_DIR.resolve()

        if dataset_root not in destination.parents and destination != dataset_root:
            raise ValueError(f"Ruta insegura dentro del ZIP: {member.filename}")

        if member.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
            continue

        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member, "r") as source, destination.open("wb") as target:
            shutil.copyfileobj(source, target)

print("Dataset extraído en:", DATASET_DIR)


Dataset extraído en: /content/seedcare/dataset


## 8. Validar imágenes y `metadata.jsonl`

In [8]:
import json
from collections import Counter

IMAGES_DIR = DATASET_DIR / "images"
METADATA_FILE = DATASET_DIR / "metadata.jsonl"
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

assert IMAGES_DIR.exists(), "No se encontró images/."
assert METADATA_FILE.exists(), "No se encontró metadata.jsonl."

images = sorted(
    path
    for path in IMAGES_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS
)

records = []
with METADATA_FILE.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(
                f"JSON inválido en metadata.jsonl, línea {line_number}: {exc}"
            ) from exc

assert images, "No se encontraron imágenes."
assert len(records) == len(images), (
    f"Metadatos: {len(records)}; imágenes: {len(images)}"
)

missing_files = []
invalid_captions = []

for record in records:
    file_name = record.get("file_name", "")
    caption = record.get("text", "")
    image_path = DATASET_DIR / file_name

    if not image_path.exists():
        missing_files.append(file_name)

    if not caption or "soyseed" not in caption.lower():
        invalid_captions.append(file_name)

assert not missing_files, f"Archivos faltantes: {missing_files[:10]}"
assert not invalid_captions, (
    f"Captions vacíos o sin 'soyseed': {invalid_captions[:10]}"
)

class_counts = Counter(path.name.split("_")[0] for path in images)

print("Imágenes:", len(images))
print("Registros:", len(records))
print("Distribución aproximada por prefijo:")
for label, count in sorted(class_counts.items()):
    print(f"  {label}: {count}")

print("Dataset validado correctamente.")


Imágenes: 1000
Registros: 1000
Distribución aproximada por prefijo:
  broken: 200
  immature: 200
  intact: 200
  skin: 200
  spotted: 200
Dataset validado correctamente.


## 9. Función común para ejecutar el entrenamiento

In [9]:
def run_lora_training(
    *,
    output_dir: Path,
    max_train_steps: int,
    checkpointing_steps: int,
    validation: bool,
    resume_from_checkpoint: str | None = None,
    use_8bit_adam: bool = False,
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    command = [
        "accelerate",
        "launch",
        str(TRAIN_SCRIPT),
        f"--pretrained_model_name_or_path={MODEL_ID}",
        f"--train_data_dir={DATASET_DIR}",
        "--image_column=image",
        "--caption_column=text",
        "--resolution=512",
        "--random_flip",
        "--train_batch_size=1",
        "--gradient_accumulation_steps=4",
        f"--max_train_steps={max_train_steps}",
        "--learning_rate=1e-4",
        "--rank=8",
        "--lr_scheduler=constant",
        "--lr_warmup_steps=0",
        "--mixed_precision=fp16",
        "--gradient_checkpointing",
        f"--checkpointing_steps={checkpointing_steps}",
        "--checkpoints_total_limit=2",
        "--report_to=tensorboard",
        f"--logging_dir={output_dir / 'logs'}",
        f"--output_dir={output_dir}",
        "--seed=42",
    ]

    if validation:
        command.extend(
            [
                (
                    "--validation_prompt=photo of a soyseed soybean seed, "
                    "broken physical damage, centered inspection photography"
                ),
                "--num_validation_images=2",
                "--validation_epochs=1",
            ]
        )

    if resume_from_checkpoint:
        command.append(
            f"--resume_from_checkpoint={resume_from_checkpoint}"
        )

    if use_8bit_adam:
        command.append("--use_8bit_adam")

    print("Ejecutando:")
    print(" ".join(map(str, command)))
    subprocess.run(command, check=True)


## 10. Smoke test de 50 pasos

In [10]:
if SMOKE_OUTPUT_DIR.exists():
    shutil.rmtree(SMOKE_OUTPUT_DIR)

run_lora_training(
    output_dir=SMOKE_OUTPUT_DIR,
    max_train_steps=50,
    checkpointing_steps=25,
    validation=False,
)

smoke_weights = SMOKE_OUTPUT_DIR / WEIGHT_NAME
assert smoke_weights.exists(), (
    f"El smoke test terminó, pero no generó: {smoke_weights}"
)

print("Smoke test correcto:", smoke_weights)
print("Tamaño MB:", round(smoke_weights.stat().st_size / 1024**2, 2))


Ejecutando:
accelerate launch /content/seedcare/diffusers/examples/text_to_image/train_text_to_image_lora.py --pretrained_model_name_or_path=stable-diffusion-v1-5/stable-diffusion-v1-5 --train_data_dir=/content/seedcare/dataset --image_column=image --caption_column=text --resolution=512 --random_flip --train_batch_size=1 --gradient_accumulation_steps=4 --max_train_steps=50 --learning_rate=1e-4 --rank=8 --lr_scheduler=constant --lr_warmup_steps=0 --mixed_precision=fp16 --gradient_checkpointing --checkpointing_steps=25 --checkpoints_total_limit=2 --report_to=tensorboard --logging_dir=/content/seedcare/outputs/smoke_test/logs --output_dir=/content/seedcare/outputs/smoke_test --seed=42
Smoke test correcto: /content/seedcare/outputs/smoke_test/pytorch_lora_weights.safetensors
Tamaño MB: 6.12


## 11. Entrenamiento completo de 1200 pasos

- La primera vez, deja `RESUME_FROM_LATEST = False`.
- Si Colab se interrumpe y existen `checkpoint-*`, cambia a `True`.
- Si aparece falta de memoria, cambia `USE_8BIT_ADAM` a `True`.


In [11]:
RESUME_FROM_LATEST = False
USE_8BIT_ADAM = False

resume_value = "latest" if RESUME_FROM_LATEST else None

run_lora_training(
    output_dir=LORA_OUTPUT_DIR,
    max_train_steps=1200,
    checkpointing_steps=250,
    validation=True,
    resume_from_checkpoint=resume_value,
    use_8bit_adam=USE_8BIT_ADAM,
)

LORA_WEIGHTS = LORA_OUTPUT_DIR / WEIGHT_NAME

assert LORA_WEIGHTS.exists(), (
    "El entrenamiento no produjo el adaptador final. "
    f"No existe: {LORA_WEIGHTS}"
)

print("Entrenamiento completo terminado.")
print("Adaptador:", LORA_WEIGHTS)
print("Tamaño MB:", round(LORA_WEIGHTS.stat().st_size / 1024**2, 2))


Ejecutando:
accelerate launch /content/seedcare/diffusers/examples/text_to_image/train_text_to_image_lora.py --pretrained_model_name_or_path=stable-diffusion-v1-5/stable-diffusion-v1-5 --train_data_dir=/content/seedcare/dataset --image_column=image --caption_column=text --resolution=512 --random_flip --train_batch_size=1 --gradient_accumulation_steps=4 --max_train_steps=1200 --learning_rate=1e-4 --rank=8 --lr_scheduler=constant --lr_warmup_steps=0 --mixed_precision=fp16 --gradient_checkpointing --checkpointing_steps=250 --checkpoints_total_limit=2 --report_to=tensorboard --logging_dir=/content/seedcare/outputs/soybean_sd15_lora/logs --output_dir=/content/seedcare/outputs/soybean_sd15_lora --seed=42 --validation_prompt=photo of a soyseed soybean seed, broken physical damage, centered inspection photography --num_validation_images=2 --validation_epochs=1
Entrenamiento completo terminado.
Adaptador: /content/seedcare/outputs/soybean_sd15_lora/pytorch_lora_weights.safetensors
Tamaño MB: 6.

## 12. Guardar el adaptador en Google Drive

In [12]:
assert LORA_OUTPUT_DIR.exists(), (
    f"No existe la carpeta de salida: {LORA_OUTPUT_DIR}"
)
assert LORA_WEIGHTS.exists(), (
    f"No existe el archivo final: {LORA_WEIGHTS}"
)

shutil.copytree(
    LORA_OUTPUT_DIR,
    DRIVE_MODEL_DIR,
    dirs_exist_ok=True,
)

drive_weights = DRIVE_MODEL_DIR / WEIGHT_NAME

assert drive_weights.exists(), (
    f"No se copió el adaptador a Drive: {drive_weights}"
)

print("Adaptador guardado permanentemente en:")
print(drive_weights)
print("Tamaño MB:", round(drive_weights.stat().st_size / 1024**2, 2))


Adaptador guardado permanentemente en:
/content/drive/MyDrive/SeedCare-RAG-LoRA/models/soybean_sd15_lora/pytorch_lora_weights.safetensors
Tamaño MB: 6.12


## 13. Comparar modelo base contra modelo con LoRA

Las tuberías se cargan de forma secuencial para evitar mantener dos copias completas de Stable Diffusion en la GPU T4 al mismo tiempo.


In [16]:
import gc
from PIL import Image, ImageDraw
from diffusers import StableDiffusionPipeline

PROMPTS = {
    "intact": (
        "photo of a soyseed soybean seed, intact condition, "
        "clean surface, centered inspection photography"
    ),
    "spotted": (
        "photo of a soyseed soybean seed, visible spotted surface, "
        "centered inspection photography"
    ),
    "immature": (
        "photo of a soyseed soybean seed, immature condition, "
        "incomplete visual development, centered inspection photography"
    ),
    "broken": (
        "photo of a soyseed soybean seed, broken physical damage, "
        "visible fractured structure, centered inspection photography"
    ),
    "skin_damaged": (
        "photo of a soyseed soybean seed, damaged outer skin, "
        "visible surface damage, centered inspection photography"
    ),
}

BASE_DIR = COMPARISON_DIR / "base"
LORA_DIR = COMPARISON_DIR / "lora"
GRIDS_DIR = COMPARISON_DIR / "grids"

for directory in [BASE_DIR, LORA_DIR, GRIDS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def generate_set(
    pipeline: StableDiffusionPipeline,
    output_dir: Path,
) -> None:
    for index, (label, prompt) in enumerate(PROMPTS.items()):
        generator = torch.Generator(device="cuda").manual_seed(42 + index)
        image = pipeline(
            prompt,
            generator=generator,
            num_inference_steps=30,
            guidance_scale=7.5,
        ).images[0]
        image.save(output_dir / f"{label}_seed{42 + index}.png")

# Generar línea base.
base_pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
).to("cuda")
base_pipe.enable_attention_slicing()

generate_set(base_pipe, BASE_DIR)

del base_pipe
gc.collect()
torch.cuda.empty_cache()

# Generar con LoRA.
lora_pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
).to("cuda")
lora_pipe.enable_attention_slicing()

lora_pipe.load_lora_weights(
    str(DRIVE_MODEL_DIR),
    weight_name=WEIGHT_NAME,
)

generate_set(lora_pipe, LORA_DIR)

del lora_pipe
gc.collect()
torch.cuda.empty_cache()

print("Generaciones terminadas.")


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


Generaciones terminadas.


## 14. Crear comparaciones lado a lado

In [17]:
def side_by_side(
    left_path: Path,
    right_path: Path,
    title: str,
) -> Image.Image:
    left = Image.open(left_path).convert("RGB")
    right = Image.open(right_path).convert("RGB")

    width = left.width + right.width
    header_height = 45
    canvas = Image.new(
        "RGB",
        (width, max(left.height, right.height) + header_height),
        "white",
    )
    canvas.paste(left, (0, header_height))
    canvas.paste(right, (left.width, header_height))

    draw = ImageDraw.Draw(canvas)
    draw.text((10, 10), f"{title}: SD 1.5 base", fill="black")
    draw.text(
        (left.width + 10, 10),
        f"{title}: SD 1.5 + LoRA",
        fill="black",
    )
    return canvas

for index, label in enumerate(PROMPTS):
    base_path = BASE_DIR / f"{label}_seed{42 + index}.png"
    lora_path = LORA_DIR / f"{label}_seed{42 + index}.png"

    comparison = side_by_side(base_path, lora_path, label)
    comparison.save(GRIDS_DIR / f"{label}_comparison.png")

shutil.copytree(
    COMPARISON_DIR,
    DRIVE_COMPARISON_DIR,
    dirs_exist_ok=True,
)

print("Comparaciones guardadas en:")
print(DRIVE_COMPARISON_DIR)


Comparaciones guardadas en:
/content/drive/MyDrive/SeedCare-RAG-LoRA/outputs/comparisons


## 15. Guardar manifiesto de ejecución

In [18]:
from datetime import datetime, timezone

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "trigger_word": "soyseed",
    "dataset_images": len(images),
    "max_train_steps": 1200,
    "rank": 8,
    "learning_rate": 1e-4,
    "train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "resolution": 512,
    "seed": 42,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "weights_path": str(DRIVE_MODEL_DIR / WEIGHT_NAME),
}

manifest_path = DRIVE_MODEL_DIR / "run_manifest.json"

with manifest_path.open("w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2, ensure_ascii=False)

print(json.dumps(manifest, indent=2, ensure_ascii=False))


{
  "created_at_utc": "2026-07-13T05:03:02.644160+00:00",
  "model_id": "stable-diffusion-v1-5/stable-diffusion-v1-5",
  "trigger_word": "soyseed",
  "dataset_images": 1000,
  "max_train_steps": 1200,
  "rank": 8,
  "learning_rate": 0.0001,
  "train_batch_size": 1,
  "gradient_accumulation_steps": 4,
  "resolution": 512,
  "seed": 42,
  "torch": "2.11.0+cu128",
  "cuda": "12.8",
  "gpu": "Tesla T4",
  "weights_path": "/content/drive/MyDrive/SeedCare-RAG-LoRA/models/soybean_sd15_lora/pytorch_lora_weights.safetensors"
}


## 16. Resultado esperado

Al finalizar deben existir:

- `pytorch_lora_weights.safetensors` en Google Drive.
- Checkpoints de entrenamiento.
- Logs de TensorBoard.
- Cinco imágenes del modelo base.
- Cinco imágenes del modelo con LoRA.
- Cinco comparaciones lado a lado.
- `run_manifest.json` con la configuración y versiones.

El siguiente paso del proyecto será revisar las imágenes sintéticas, seleccionar las válidas y preparar el Experimento B del clasificador.
